# UCell pathway scoring for 10x Atera whole-transcriptome spatial data
### Human cervical cancer FFPE sample — single-cell-resolution spatial transcriptomics

This notebook takes a **single-cell-resolution spatial** dataset (10x Atera whole-transcriptome
output, same shape as Xenium: one row per *cell*, not per Visium spot) and produces a
**pathway-activity table**: for every cell, a UCell score for each of a set of MSigDB
pathways. That table is the training-label file for the downstream **iSCALE** deep-learning model.

**UCell** (Andreatta & Carmona) scores a gene signature in one cell using the Mann–Whitney U
statistic on the *within-cell gene ranking*. Because it is rank-based it is robust to
sequencing depth and does not need the data to be batch-corrected or scaled.

---

## What each step does (plain language)

| Step | What happens | Why |
|---|---|---|
| **0. Upload** | You upload `cell_feature_matrix.h5` and `cells.csv` through the browser. The cell checks both arrived with the right names. | Colab has no access to your disk; files come in as a dict keyed by original filename. |
| **1. Parameters** | All tunable cutoffs are set in one cell. | So you can change a threshold and re-run without hunting through the code. |
| **2. Install / load** | Install `scanpy`, `gseapy`, `pyucell`; read the count matrix and the cell metadata; match every cell in the matrix to its (x, y) centroid. | UCell needs the expression matrix; the spatial plots and the final CSV need the coordinates. Cells that can't be matched are reported and dropped, never silently kept. |
| **3. QC + normalize** | Drop near-empty **cells** (cheap row filter). Genes are *not* dropped from the matrix — a per-gene detection count is recorded instead and applied in Step 5. UCell runs on the raw-counts layer (see the note below the QC cells). | Removes technical noise without any gene-axis matrix op (those cause the OOM crash). |
| **4. Gene sets** | Pull the **Hallmark (h.all)** collection and a **curated slice of the C6 oncogenic (c6.all)** collection from MSigDB via `gseapy`. The MSigDB version is chosen automatically. The C6 picks (HPV E6/E7 → p53 loss & RB/E2F release, PI3K–AKT–mTOR) are printed so you can edit the list. |
| **5. Coverage filter** | Build the list of *usable* genes (in the panel, real gene not control probe, detected in ≥ `MIN_CELLS_PER_GENE` cells). Intersect each gene set with it; drop the whole set if too little survives. Before/after size table printed. | A signature with 3/200 genes present — or present only as silent/control features — is not meaningful. |
| **6. Run UCell** | `pyucell.compute_ucell_scores` → one score per pathway per cell, added to `adata.obs`. Runs **single-process** (`n_jobs=1`) in modest cell chunks — multi-process would pickle a full AnnData copy per worker and OOM. Sequential, so a ~700k-cell run takes a few minutes. |
| **7. Score QC + spatial plots** | Drop pathways whose score barely varies across cells; plot each surviving pathway as a spatial scatter (x, y colored by score) in a subplot grid. | A real pathway signal is spatially structured (tumor nests, stroma, immune aggregates); pure noise is not. |
| **8. Save** | Write one tidy CSV: `cell_id, x, y, UCell_<pathway>...`, print the final shape and the list of retained pathways. | This is the label table iSCALE consumes. |

> **Re-running:** every cutoff that changes the biology lives in the **Parameters** cell
> (Step 1). Change it there, then run the cells below it again.

> **Memory (free Colab ≈ 12 GB).** This matrix is ~5 GB sparse; it fits, but only if nothing
> ever subsets or copies its **gene axis** (that briefly doubles it). So Step 2 reads the
> HDF5 arrays directly, builds the matrix with no transpose/copy, and **keeps every gene** —
> control-probe and low-count genes are excluded later in Step 5 on the gene-set side, which
> is cheap. Every heavy step prints process RAM. If it still crashes: use a **High-RAM**
> runtime, lower `UCELL_CHUNK_SIZE`, or set `SUBSAMPLE_N_CELLS` in the Step 2 loader cell.


## Step 0 — Upload the data files

Run this cell, then pick **both** files in the dialog:

* `cell_feature_matrix.h5` — 10x-format HDF5, cells × genes **raw counts**
* `cells.csv` **or** `cells.csv.gz` — per-cell metadata: a cell-id column and centroid
  x/y in image-pixel space (gzip is accepted; pandas reads it transparently)

The cell warns loudly if a filename doesn't match what the rest of the notebook expects
(the upload dict is keyed by the *original* filename, so a renamed file breaks the pipeline).


In [ ]:
from google.colab import files

uploaded = files.upload()
received = set(uploaded.keys())

print("\nReceived:", sorted(received))

# --- count matrix (required, exact name) ---
H5_PATH = "cell_feature_matrix.h5"

# --- cell metadata (required, plain OR gzipped) ---
CELLS_CSV_CANDIDATES = ["cells.csv", "cells.csv.gz"]
CELLS_CSV_PATH = next((f for f in CELLS_CSV_CANDIDATES if f in received), None)

expected_any = {H5_PATH, *CELLS_CSV_CANDIDATES}
missing = []
if H5_PATH not in received:
    missing.append(H5_PATH)
    print(f"  !! MISSING expected file: {H5_PATH!r}")
if CELLS_CSV_PATH is None:
    missing.append("cells.csv (or cells.csv.gz)")
    print(f"  !! MISSING expected file: one of {CELLS_CSV_CANDIDATES}")
for f in sorted(received - expected_any):
    print(f"  !! UNEXPECTED filename (will be ignored): {f!r}  "
          f"-> rename it to {H5_PATH!r} or one of {CELLS_CSV_CANDIDATES} and re-run")

if missing:
    raise FileNotFoundError(
        "Upload incomplete: " + ", ".join(missing) +
        ". Re-run this cell and select the correctly named file(s)."
    )

print(f"\nOK - count matrix : {H5_PATH}")
print(f"OK - cell metadata: {CELLS_CSV_PATH}")


## Step 1 — Tunable parameters

Everything that affects the biological result is here. Edit, then re-run the cells below.


In [ ]:
# =========================== TUNABLE PARAMETERS ============================

# ---- Step 3: per-cell / per-gene QC ----
MIN_COUNTS_PER_CELL = 10     # drop cells with fewer total transcripts than this
MIN_GENES_PER_CELL  = 3      # drop cells expressing fewer distinct genes than this
MIN_CELLS_PER_GENE  = 5      # drop genes detected in fewer cells than this

# ---- Step 3: normalization ----
NORMALIZE_TARGET_SUM = 1e4   # counts-per-cell target for sc.pp.normalize_total

# ---- Step 5: gene-set coverage filter ----
MIN_GENE_SET_COVERAGE = 0.30 # keep a set only if >= this fraction of its genes are in the panel
MIN_GENES_RETAINED    = 5    # ...AND at least this many genes remain after intersect

# ---- Step 6: UCell ----
UCELL_LAYER      = "counts"  # AnnData layer UCell ranks: "counts" (raw) or "norm" (lib-size norm)
UCELL_MAX_RANK   = 1500      # pyucell default; genes ranked worse than this are ignored
UCELL_CHUNK_SIZE = 10_000    # cells scored per chunk; lower to ~2000 if UCell still OOMs
UCELL_N_JOBS     = 1         # KEEP AT 1 on Colab: n_jobs>1 pickles a full AnnData copy per
                             # worker process (loky backend) -> out-of-RAM on a 2-core VM

# ---- Step 7: pathway-score variance filter ----
MIN_SCORE_VARIANCE = 1e-4    # drop UCell pathways whose across-cell score variance is below this

# ---- Step 4: MSigDB ----
# C6 oncogenic-signature name tokens relevant to cervical / HPV biology.
#   E6 -> TP53 degradation ; E7 -> RB1 inactivation -> E2F / cell cycle ; frequent PIK3CA activation
C6_NAME_TOKENS = [
    "E2F", "RB_", "RB1", "P53", "TP53",
    "CYCLIN_D1", "CCND1", "CYCLIN",
    "PI3K", "PIK3", "AKT", "MTOR", "PTEN",
]

# ---- cells.csv column autodetection (first match wins; add your file's names if needed) ----
CELL_ID_CANDIDATES = ["cell_id", "cell_ID", "Cell_ID", "id", "cell"]
X_CANDIDATES       = ["x_centroid", "x", "centroid_x", "x_pixel", "x_px", "pxl_col_in_fullres"]
Y_CANDIDATES       = ["y_centroid", "y", "centroid_y", "y_pixel", "y_px", "pxl_row_in_fullres"]

# ---- output ----
OUTPUT_CSV = "ucell_pathway_scores.csv"

# =========================================================================
print("Parameters loaded.")


## Step 2 — Install dependencies

In [ ]:
%pip install --quiet scanpy gseapy pyucell
print("install done")


In [ ]:
import warnings
import re
import gc

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import h5py
import scipy.sparse as sp
import anndata as ad
import scanpy as sc
import gseapy
from gseapy import Msigdb
import pyucell as uc

try:
    import psutil
    def ram_gb():
        return psutil.Process().memory_info().rss / 1e9
except Exception:
    def ram_gb():
        return float("nan")

sc.settings.verbosity = 1
warnings.simplefilter("ignore", category=FutureWarning)

print("scanpy ", sc.__version__)
print("gseapy ", gseapy.__version__)
print("pyucell", getattr(uc, "__version__", "(no __version__)"))
print(f"process RAM now: {ram_gb():.2f} GB")


## Step 2 (cont.) — Load the matrix and the cell metadata, and align them

**Why the earlier versions crashed.** `717,576 cells x 27,104 genes` is ~5 GB as a sparse
matrix. That fits in free Colab (~12 GB) — but **any operation that subsets or copies the
gene axis briefly doubles it** past the ceiling. `scanpy.read_10x_h5` + the control-probe
filter (`adata[:, mask].copy()` / `_inplace_subset_var`, a CSR→CSC copy) is exactly such an
operation.

**The fix here: never touch the gene axis of the matrix.**

* Read the HDF5 arrays directly with `h5py`; the 10x CSC(genes×cells) triplet is bit-identical
  to a CSR(cells×genes) triplet, so we build `X` with **no transpose and no copy**.
* Sparse indices downcast to `int32` when possible.
* **Keep all 27,104 genes in `X`.** Control probes and low-count genes are handled *logically*
  in Step 5 (they simply won't match a pathway gene list), not by restructuring the matrix.
* Cell-axis operations (QC filters, coordinate alignment, subsampling) all go through
  `subset_cells()`, which **returns the object untouched when nothing is dropped** and
  otherwise gathers kept rows in small chunks into pre-sized arrays — no second full copy
  of the matrix, no `scanpy`/`anndata` `_inplace_subset_obs` (which secretly does
  `self[mask].copy()` and doubles RAM even for an all-keep mask — that was the last crash).
* Every heavy step prints process RAM.

**A 717k-cell full run genuinely sits near the free-Colab ceiling.** If any step still
crashes: use a **High-RAM runtime** (*Runtime ▸ Change runtime type ▸ High-RAM*), lower
`UCELL_CHUNK_SIZE`, or set `SUBSAMPLE_N_CELLS` below.


In [ ]:
# Optional emergency valve: cap the number of cells loaded (None = load all).
SUBSAMPLE_N_CELLS = None   # e.g. 400_000 to fit a small VM; None = full run


def subset_cells(adata, keep_bool, chunk=100_000):
    """Drop cells (rows) with minimal peak RAM.

    * If keep_bool is all-True -> returns adata untouched (0 copies).
    * Otherwise builds the new CSR by gathering kept rows in small chunks into
      pre-sized arrays: no whole-matrix second copy, no CSR<->CSC conversion.
    """
    keep_bool = np.asarray(keep_bool, dtype=bool)
    if keep_bool.all():
        return adata

    keep_idx = np.flatnonzero(keep_bool)
    X = adata.X
    if not sp.isspmatrix_csr(X):
        X = X.tocsr()
    nnz_per_row = np.diff(X.indptr)
    out_nnz = int(nnz_per_row[keep_idx].sum())

    nd  = np.empty(out_nnz, X.data.dtype)
    ni  = np.empty(out_nnz, X.indices.dtype)
    npr = np.empty(keep_idx.size + 1, dtype=np.int64)
    npr[0] = 0
    w = 0
    for s in range(0, keep_idx.size, chunk):
        rs  = keep_idx[s:s + chunk]
        blk = X[rs]                      # small: <= `chunk` rows
        m   = blk.data.size
        nd[w:w + m] = blk.data
        ni[w:w + m] = blk.indices
        npr[s + 1:s + 1 + rs.size] = w + blk.indptr[1:]
        w += m
        del blk
    new_X = sp.csr_matrix((nd, ni, npr), shape=(keep_idx.size, X.shape[1]))
    out = ad.AnnData(X=new_X, obs=adata.obs.iloc[keep_idx].copy(), var=adata.var)
    for k in list(adata.layers.keys()):             # layers here only ever alias X
        out.layers[k] = new_X
    del X, nd, ni, npr
    gc.collect()
    return out


def load_10x_h5_lean(path, subsample_n=None, seed=0):
    """Low-memory 10x .h5 reader -> AnnData (cells x genes), CSR, ALL genes kept."""
    with h5py.File(path, "r") as f:
        grp = "matrix" if "matrix" in f else list(f.keys())[0]
        g = f[grp]
        data    = g["data"][:]
        indices = g["indices"][:]                     # 10x CSC(genes x cells) == CSR(cells x genes):
        indptr  = g["indptr"][:]                      #   indptr over cells, indices = gene id
        n_genes, n_cells = (int(v) for v in g["shape"][:])
        barcodes = g["barcodes"][:].astype(str)
        if "features" in g:                           # 10x v3
            feat = g["features"]
            gene_names = feat["name"][:].astype(str)
            gene_ids   = feat["id"][:].astype(str)
            ftype = feat["feature_type"][:].astype(str) if "feature_type" in feat else None
        else:                                         # 10x v1/v2
            gene_names = g["gene_names"][:].astype(str)
            gene_ids   = g["genes"][:].astype(str)
            ftype = None

    nnz = int(indptr[-1])
    print(f"  {n_cells:,} cells x {n_genes:,} genes, {nnz:,} non-zeros "
          f"(~{nnz * 8 / 1e9:.1f} GB sparse)")
    data = data.astype(np.float32, copy=False)          # 4 bytes either way; needed by pyucell/scanpy
    if nnz < np.iinfo(np.int32).max:
        indices = indices.astype(np.int32, copy=False)
        indptr  = indptr.astype(np.int32, copy=False)

    X = sp.csr_matrix((data, indices, indptr), shape=(n_cells, n_genes))   # no transpose, no copy
    del data, indices, indptr; gc.collect()

    var = pd.DataFrame({"gene_ids": gene_ids}, index=pd.Index(gene_names))
    if ftype is not None:
        var["feature_types"] = ftype
    adata = ad.AnnData(X=X, obs=pd.DataFrame(index=barcodes), var=var)
    adata.var_names_make_unique()

    if subsample_n is not None and subsample_n < adata.n_obs:
        rng = np.random.default_rng(seed)
        m = np.zeros(adata.n_obs, dtype=bool)
        m[rng.choice(adata.n_obs, size=subsample_n, replace=False)] = True
        adata = subset_cells(adata, m)
        gc.collect()
        print(f"  subsampled to {adata.n_obs:,} cells")
    return adata


print(f"RAM before load: {ram_gb():.2f} GB")
try:
    adata = load_10x_h5_lean(H5_PATH, subsample_n=SUBSAMPLE_N_CELLS)
except Exception as e:
    print(f"lean reader failed ({e!r}); falling back to scanpy.read_10x_h5")
    adata = sc.read_10x_h5(H5_PATH)
    adata.var_names_make_unique()

adata.obs_names = adata.obs_names.astype(str)
gc.collect()
print(f"Raw matrix (cells x genes): {adata.shape}   |  RAM: {ram_gb():.2f} GB")

# Cheap per-gene detection count (bincount on the existing index array; no matrix copy).
adata.var["n_cells_by_counts"] = np.bincount(adata.X.indices, minlength=adata.n_vars)
if "feature_types" in adata.var.columns:
    vc = adata.var["feature_types"].value_counts()
    print("\nfeature_types present:", dict(vc))
    print("(all genes kept in the matrix; non-'Gene Expression' features are excluded "
          "in Step 5)")

cells_df = pd.read_csv(CELLS_CSV_PATH)
print("\ncells.csv:", cells_df.shape)
print("cells.csv columns:", list(cells_df.columns))
cells_df.head()


In [ ]:
def _pick(columns, candidates, what):
    for c in candidates:
        if c in columns:
            return c
    raise KeyError(
        f"No {what} column found in cells.csv. Looked for {candidates}, "
        f"file has {list(columns)}. Add the right name to the *_CANDIDATES list in Step 1."
    )

cid_col = _pick(cells_df.columns, CELL_ID_CANDIDATES, "cell-id")
x_col   = _pick(cells_df.columns, X_CANDIDATES, "x-centroid")
y_col   = _pick(cells_df.columns, Y_CANDIDATES, "y-centroid")
print(f"Using  cell_id={cid_col!r}  x={x_col!r}  y={y_col!r}")

cells_df[cid_col] = cells_df[cid_col].astype(str)
coords = (cells_df.set_index(cid_col)[[x_col, y_col]]
                  .rename(columns={x_col: "x", y_col: "y"}))
coords = coords[~coords.index.duplicated(keep="first")]

csv_ids  = set(coords.index)
mat_ids  = set(adata.obs_names)
keep_mask = adata.obs_names.isin(csv_ids)          # numpy bool over matrix cells
n_dropped = int((~keep_mask).sum())

print(f"\ncells in matrix            : {adata.n_obs:>8}")
print(f"cells in cells.csv         : {len(csv_ids):>8}")
print(f"matrix cells w/o centroid  : {n_dropped:>8}   <- DROPPED")
print(f"cells.csv rows not in matrix: {len(csv_ids - mat_ids):>7}   <- ignored")
del mat_ids, csv_ids

adata = subset_cells(adata, keep_mask)             # no-op if nothing to drop
gc.collect()

c = coords.reindex(adata.obs_names)                # align by cell id, same order as adata
adata.obs["cell_id"] = adata.obs_names.to_numpy()
adata.obs["x"] = c["x"].to_numpy()
adata.obs["y"] = c["y"].to_numpy()
del c, coords, cells_df; gc.collect()

assert adata.obs[["x", "y"]].notna().all().all(), "Some kept cells still have NaN coordinates"
print(f"\nAligned AnnData: {adata.shape}   ({n_dropped} cells dropped)   |  RAM: {ram_gb():.2f} GB")


## Step 3 — QC and normalization

In [ ]:
# CELL-axis QC only. Metrics are computed cheaply; the matrix is rebuilt at most ONCE
# (via subset_cells), and not at all if no cell fails the thresholds.
n0 = adata.n_obs
counts_per_cell = np.asarray(adata.X.sum(axis=1)).ravel()
genes_per_cell  = adata.X.getnnz(axis=1)

cell_ok = (counts_per_cell >= MIN_COUNTS_PER_CELL) & (genes_per_cell >= MIN_GENES_PER_CELL)
print(f"cells failing QC: {int((~cell_ok).sum()):,} / {n0:,}")
print(f"median counts/cell: {np.median(counts_per_cell):.0f}   |  RAM: {ram_gb():.2f} GB")
del counts_per_cell, genes_per_cell

adata = subset_cells(adata, cell_ok)
gc.collect()

# per-gene detection count (cheap bincount; no matrix copy) -- used by Step 5
adata.var["n_cells_by_counts"] = np.bincount(adata.X.indices, minlength=adata.n_vars)
n_genes_pass = int((adata.var["n_cells_by_counts"] >= MIN_CELLS_PER_GENE).sum())

print(f"\ncells: {n0:>9}  ->  {adata.n_obs:>9}")
print(f"genes in matrix (all kept):                       {adata.n_vars:>9}")
print(f"genes detected in >= {MIN_CELLS_PER_GENE} cells (usable in Step 5): {n_genes_pass:>9}")
print(f"RAM: {ram_gb():.2f} GB")


In [ ]:
# No .copy() of the matrix here -- that would double ~5 GB and crash the VM.
if UCELL_LAYER == "counts":
    adata.layers["counts"] = adata.X                 # same array; X is not modified after this
elif UCELL_LAYER == "norm":
    sc.pp.normalize_total(adata, target_sum=NORMALIZE_TARGET_SUM)   # in place on X
    adata.layers["norm"] = adata.X
else:
    raise ValueError(f"UCELL_LAYER must be 'counts' or 'norm', got {UCELL_LAYER!r}")
gc.collect()

print("layers:", list(adata.layers.keys()), " | UCell will read:", repr(UCELL_LAYER))
print(f"X dtype: {adata.X.dtype}   |  RAM: {ram_gb():.2f} GB")


### Which layer does UCell use, and why

UCell scores a signature from the **rank of each gene within a cell**, via the Mann–Whitney U
statistic. Consequences:

* **Monotonic transforms don't matter.** `log1p` is monotonic, so log vs non-log gives the
  *same* per-cell ranking and the *same* UCell score.
* **Library-size normalization barely matters** either — it rescales every gene in a cell by
  the same factor, so the ranking is unchanged (only tie-breaking at zero counts can differ).

The pyUCell tutorial computes scores on the **raw count matrix, before** `normalize_total` /
`log1p`. This notebook follows that: with `UCELL_LAYER = "counts"` (the default) the raw
post-QC counts are used and **no copy or normalization of the matrix happens at all** — the
`counts` layer just points at `adata.X`. Set `UCELL_LAYER = "norm"` in Step 1 to library-size
normalize `adata.X` in place first; scores should be near-identical.

Nothing downstream reads `adata.X` numerically (the spatial plots and the output CSV read
`adata.obs`), so no log-normalized layer is built. If you want one for your own PCA/UMAP
work, add `sc.pp.normalize_total(adata); sc.pp.log1p(adata)` in a fresh cell after Step 8.


## Step 4 — Retrieve MSigDB gene sets (`gseapy.Msigdb`)

* **Hallmark** (`h.all`), human, newest available DB version — chosen with `list_dbver()`,
  not hardcoded.
* **Curated C6 oncogenic** subset (`c6.all`) — selected by name-matching against
  `C6_NAME_TOKENS` (Step 1). The full selected list is printed for you to review/edit.


In [ ]:
msig = Msigdb()

vers = msig.list_dbver()
print("MSigDB versions reported by gseapy:")
print(vers)

def _pick_latest_human(vers):
    if hasattr(vers, "columns"):                       # DataFrame
        col = "db_ver" if "db_ver" in vers.columns else vers.columns[0]
        cand = [str(v) for v in vers[col].tolist()]
    else:                                              # list / iterable
        cand = [str(v) for v in vers]
    human = [v for v in cand if v.endswith(".Hs")]
    human = [v for v in human if "msigdb" not in v.lower()] or human
    if not human:
        raise RuntimeError(f"No human (.Hs) MSigDB version in {cand}")
    def key(v):
        core = v.replace("msigdb_v", "").replace(".Hs", "")
        return [int(p) if p.isdigit() else p for p in core.split(".")]
    return sorted(human, key=key)[-1]

DBVER = _pick_latest_human(vers)
print("\n--> using MSigDB version:", DBVER)
print("categories available:", msig.list_category(dbver=DBVER))


In [ ]:
hallmark = msig.get_gmt(category="h.all", dbver=DBVER)
if not hallmark:
    raise RuntimeError("gseapy returned an empty Hallmark collection - check network / DBVER")
print(f"Hallmark (h.all): {len(hallmark)} gene sets")


In [ ]:
c6_all = msig.get_gmt(category="c6.all", dbver=DBVER)
if not c6_all:
    raise RuntimeError("gseapy returned an empty C6 collection")
print(f"C6 oncogenic (c6.all): {len(c6_all)} gene sets available")

_pat = re.compile("|".join(C6_NAME_TOKENS))
c6_selected = {k: v for k, v in c6_all.items() if _pat.search(k.upper())}

print(f"\n--- {len(c6_selected)} C6 sets selected by name match (edit C6_NAME_TOKENS to change) ---")
for k in sorted(c6_selected):
    print(f"  {k:<35} ({len(c6_selected[k])} genes)")


In [ ]:
gene_sets_raw = {**hallmark, **c6_selected}
print(f"Total gene sets entering the coverage filter: {len(gene_sets_raw)} "
      f"({len(hallmark)} Hallmark + {len(c6_selected)} C6)")


## Step 5 — Filter gene sets to genes usable in this panel

This is where the gene-axis filtering that we deliberately kept *out* of the matrix happens
— cheaply, on the gene-set side. A gene counts as **usable** only if it is:

1. in `adata.var_names`, **and**
2. of `feature_types == "Gene Expression"` (not a control probe), if that column exists, **and**
3. detected in at least `MIN_CELLS_PER_GENE` cells (`adata.var["n_cells_by_counts"]`).

Each gene set is intersected with the usable-gene list. Keep it only if
`coverage >= MIN_GENE_SET_COVERAGE` **and** `n_usable >= MIN_GENES_RETAINED` (Step 1).
A before/after table is printed and the dropped sets are listed.


In [ ]:
usable = pd.Series(True, index=adata.var_names)
if "feature_types" in adata.var.columns:
    usable &= (adata.var["feature_types"].to_numpy() == "Gene Expression")
usable &= (adata.var["n_cells_by_counts"].to_numpy() >= MIN_CELLS_PER_GENE)
panel = set(adata.var_names[usable.to_numpy()])
print(f"usable genes: {len(panel):,} / {adata.n_vars:,} "
      f"(feature_type + detected in >= {MIN_CELLS_PER_GENE} cells)")

rows, gene_sets_filtered = [], {}
for name, genes in gene_sets_raw.items():
    genes   = sorted(set(map(str, genes)))
    present = [g for g in genes if g in panel]
    frac    = len(present) / max(len(genes), 1)
    kept    = (frac >= MIN_GENE_SET_COVERAGE) and (len(present) >= MIN_GENES_RETAINED)
    rows.append(dict(gene_set=name, n_original=len(genes),
                     n_usable=len(present), coverage=round(frac, 3), retained=kept))
    if kept:
        gene_sets_filtered[name] = present

cov_table = (pd.DataFrame(rows)
               .sort_values(["retained", "coverage"], ascending=[False, False])
               .reset_index(drop=True))
pd.set_option("display.max_rows", None)

print(f"Retained {len(gene_sets_filtered)} / {len(gene_sets_raw)} gene sets "
      f"(coverage >= {MIN_GENE_SET_COVERAGE:.0%} AND >= {MIN_GENES_RETAINED} genes)\n")

dropped_tbl = cov_table.loc[~cov_table.retained, ["gene_set", "n_original", "n_usable", "coverage"]]
if len(dropped_tbl):
    print("Dropped:")
    print(dropped_tbl.to_string(index=False))
else:
    print("Nothing dropped.")

cov_table


## Step 6 — Run UCell

`pyucell.compute_ucell_scores` adds one column per pathway to `adata.obs`, named
`<pathway>_UCell` (scores in `[0, 1]`; higher = the signature's genes sit near the top of
that cell's ranking).

**Why the last crash happened here.** pyUCell parallelises with `joblib`'s *loky* backend,
and its per-chunk worker function closes over the whole `adata`. With `n_jobs > 1`, loky
**pickles a full copy of the ~6 GB AnnData into every worker process** — on a 2-core Colab
VM that is 2 extra copies and instant OOM. Also the CPU default `chunk_size` is **500**, and
the previous notebook passed **50 000**.

Fix: `UCELL_N_JOBS = 1` (everything runs in the main process, one chunk at a time, no
pickling) and `UCELL_CHUNK_SIZE = 10 000`. Sequential, so expect a few minutes for ~700k
cells. `max_rank` keeps each chunk's rank matrix sparse (≤ `max_rank` entries per cell).


In [ ]:
assert gene_sets_filtered, "No gene sets survived Step 5 - loosen MIN_GENE_SET_COVERAGE / MIN_GENES_RETAINED"
assert UCELL_LAYER in adata.layers, f"UCELL_LAYER={UCELL_LAYER!r} not in {list(adata.layers.keys())}"

import time, inspect
_want = dict(signatures=gene_sets_filtered, layer=UCELL_LAYER, max_rank=UCELL_MAX_RANK,
            chunk_size=UCELL_CHUNK_SIZE, n_jobs=UCELL_N_JOBS, suffix="_UCell")
_ok = set(inspect.signature(uc.compute_ucell_scores).parameters)
_kw = {k: v for k, v in _want.items() if k in _ok}
for k in _want.keys() - _kw.keys():
    print(f"note: this pyucell build has no '{k}' parameter; using its default")

print(f"RAM before UCell: {ram_gb():.2f} GB  |  {adata.n_obs:,} cells x "
      f"{len(gene_sets_filtered)} sets  |  {_kw}")
_t0 = time.time()

uc.compute_ucell_scores(adata, **_kw)
gc.collect()

score_cols = [f"{name}_UCell" for name in gene_sets_filtered]
missing = [c for c in score_cols if c not in adata.obs.columns]
assert not missing, f"UCell did not create expected columns, e.g. {missing[:3]}"

# keep obs lean: store scores as float32
adata.obs[score_cols] = adata.obs[score_cols].astype(np.float32)

print(f"Computed {len(score_cols)} UCell pathway scores for {adata.n_obs:,} cells "
      f"in {time.time() - _t0:.0f}s.  RAM: {ram_gb():.2f} GB")
adata.obs[score_cols].describe().T[["mean", "std", "min", "max"]].head(10)


## Step 7 — Score QC (variance) + spatial sanity plots

1. Drop pathways whose across-cell variance `< MIN_SCORE_VARIANCE` (Step 1) — those carry
   no usable signal.
2. For every surviving pathway, a spatial scatter (`x`, `y` from `cells.csv`, colored by the
   UCell score) in one subplot of a grid. Look for spatial structure (tumor nests, stroma,
   immune aggregates) rather than salt-and-pepper noise.


In [ ]:
score_df   = adata.obs[score_cols]
variances  = score_df.var(axis=0).sort_values()
low_var    = variances[variances < MIN_SCORE_VARIANCE].index.tolist()
retained_cols = [c for c in score_cols if c not in low_var]

print(f"Near-zero-variance pathways dropped (var < {MIN_SCORE_VARIANCE:g}): {len(low_var)}")
for c in low_var:
    print(f"   {c:<40} var={variances[c]:.2e}")
print(f"\nRetained for plotting + output: {len(retained_cols)} pathways")


In [ ]:
assert retained_cols, "Every pathway was filtered as low-variance - lower MIN_SCORE_VARIANCE"

MAX_POINTS_PER_PLOT = 150_000   # display subsample only; the saved CSV keeps every cell
PLOTS_PER_FIGURE    = 24        # split into several figures so no single one is huge

rng = np.random.default_rng(0)
idx = (np.sort(rng.choice(adata.n_obs, MAX_POINTS_PER_PLOT, replace=False))
       if adata.n_obs > MAX_POINTS_PER_PLOT else np.arange(adata.n_obs))
xy = adata.obs[["x", "y"]].to_numpy()[idx]
scores_mat = adata.obs[retained_cols].to_numpy()[idx]      # (subsample x pathways) float32
print(f"plotting {len(idx):,} of {adata.n_obs:,} cells across {len(retained_cols)} pathways")

ncol = 4
for fig_start in range(0, len(retained_cols), PLOTS_PER_FIGURE):
    block = list(range(fig_start, min(fig_start + PLOTS_PER_FIGURE, len(retained_cols))))
    nrow = int(np.ceil(len(block) / ncol))
    fig, axes = plt.subplots(nrow, ncol, figsize=(3.4 * ncol, 3.2 * nrow), squeeze=False)
    for k, ci in enumerate(block):
        ax = axes[k // ncol][k % ncol]
        s = scores_mat[:, ci]
        lo, hi = np.quantile(s, [0.01, 0.99])
        sca = ax.scatter(xy[:, 0], xy[:, 1], c=s, s=1.0, cmap="viridis",
                         vmin=lo, vmax=hi if hi > lo else lo + 1e-9,
                         linewidths=0, rasterized=True)
        ax.set_title(retained_cols[ci][:-6], fontsize=7)
        ax.set_aspect("equal"); ax.invert_yaxis()
        ax.set_xticks([]); ax.set_yticks([])
        fig.colorbar(sca, ax=ax, fraction=0.046, pad=0.04)
    for k in range(len(block), nrow * ncol):
        axes[k // ncol][k % ncol].axis("off")
    fig.tight_layout()
    plt.show()
    plt.close(fig)

del scores_mat, xy; gc.collect()


## Step 8 — Save the training-label CSV

Columns: `cell_id, x, y`, then one `UCell_<pathway>` column per retained pathway.
This file is the label table for **iSCALE**.


In [ ]:
rename = {c: f"UCell_{c[:-6] if c.endswith('_UCell') else c}" for c in retained_cols}
out = (adata.obs[["cell_id", "x", "y"] + retained_cols]
       .rename(columns=rename)
       .reset_index(drop=True))
out[list(rename.values())] = out[list(rename.values())].astype(np.float32)

out.to_csv(OUTPUT_CSV, index=False, float_format="%.5f")   # scores in [0,1]; 5 dp is plenty
import os
print(f"wrote {OUTPUT_CSV}  shape={out.shape}  size={os.path.getsize(OUTPUT_CSV)/1e6:.1f} MB")
out.head()


In [ ]:
# Optional: pull the CSV to your machine. Large files (100s of MB) can fail to download
# via the browser widget; if so, save to Google Drive instead (code below, commented).
from google.colab import files as _files
try:
    _files.download(OUTPUT_CSV)
except Exception as e:
    print(f"browser download failed ({e!r}). Alternative - mount Drive and copy:")
    print("    from google.colab import drive; drive.mount('/content/drive')")
    print(f"    import shutil; shutil.copy({OUTPUT_CSV!r}, '/content/drive/MyDrive/')")


## Final summary

In [ ]:
pathway_names = [f"UCell_{c[:-6] if c.endswith('_UCell') else c}" for c in retained_cols]

print("=================  FINAL OUTPUT  =================")
print(f"file            : {OUTPUT_CSV}")
print(f"n_cells         : {out.shape[0]}")
print(f"n_pathways      : {len(pathway_names)}")
print(f"table shape     : {out.shape[0]} x {len(pathway_names)}  (pathway columns only)")
print(f"UCell input     : layer '{UCELL_LAYER}', max_rank={UCELL_MAX_RANK}")
print(f"MSigDB version  : {DBVER}")
print(f"columns         : cell_id, x, y, " + ", ".join(pathway_names[:3]) + (", ..." if len(pathway_names) > 3 else ""))
print("\nRetained pathways:")
for p in pathway_names:
    print("  ", p)
